# SMILES → Canonical SMILES → Morgan Fingerprint → Tanimoto Similarity

Demo comparing two EGFR tyrosine kinase inhibitors: **gefitinib** and **erlotinib**.

SMILES strings sourced from PubChem:
- Gefitinib: [CID 123631](https://pubchem.ncbi.nlm.nih.gov/compound/Gefitinib)
- Erlotinib: [CID 176870](https://pubchem.ncbi.nlm.nih.gov/compound/Erlotinib)

Kaggle's default Python environment does not ship RDKit, so the first cell installs it.
If you're on a Kaggle Notebook with internet access enabled (Settings → Internet → On), this will work as-is.

In [ ]:
!pip install rdkit -q

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, DataStructs, rdMolDescriptors

# Two known EGFR inhibitors, SMILES sourced from PubChem (CID 123631, CID 176870)
molecules = {
    "gefitinib": "COC1=C(C=C2C(=C1)N=CN=C2NC3=CC(=C(C=C3)F)Cl)OCCCN4CCOCC4",
    "erlotinib": "COCCOC1=C(C=C2C(=C1)C(=NC=N2)NC3=CC=CC(=C3)C#C)OCCOC",
}

## Step 1: Parse SMILES, canonicalize, and sanity-check formula/molecular weight

RDKit's canonical SMILES is a deterministic string for a given molecule — useful for deduplication.
The computed formula/MW should match the PubChem record exactly; if not, something is wrong with the input SMILES.

In [ ]:
mols = {}
for name, smi in molecules.items():
    mol = Chem.MolFromSmiles(smi)
    mols[name] = mol
    canon = Chem.MolToSmiles(mol)
    formula = rdMolDescriptors.CalcMolFormula(mol)
    mw = Descriptors.MolWt(mol)
    print(f"--- {name} ---")
    print(f"  input SMILES:     {smi}")
    print(f"  canonical SMILES: {canon}")
    print(f"  formula (RDKit):  {formula}")
    print(f"  MW (RDKit):       {mw:.2f}")
    print()

## Step 2: Morgan fingerprints (ECFP4-equivalent: radius=2, 2048 bits)

Each "on" bit is a hashed local substructure. Molecules sharing a scaffold will share many bits.

In [ ]:
fps = {}
for name, mol in mols.items():
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
    fps[name] = fp
    on_bits = list(fp.GetOnBits())
    print(f"{name}: {fp.GetNumOnBits()} bits set out of 2048")
    print(f"  first 15 on-bit indices: {on_bits[:15]}")

## Step 3: Tanimoto similarity

Intersection over union of the two bit vectors — the standard structural similarity metric in cheminformatics.
Relevant later for scaffold-based train/test splitting to avoid data leakage.

In [ ]:
sim = DataStructs.TanimotoSimilarity(fps["gefitinib"], fps["erlotinib"])
print(f"Tanimoto similarity (gefitinib vs erlotinib): {sim:.3f}")

## Try it yourself

Swap in SMILES for your own targets of interest (EGFR, VEGFR2, SRC, ABL1, BRAF ligands from ChEMBL/BindingDB)
and re-run to compare fingerprints/similarity across a larger set.